# Notebook 07 — Matrices as Graphs: Adjacency and Laplacian

**Module:** 04 — Linear Algebra  
**Notebook:** 07 of 10  
**Prerequisites:** NB04  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

Biological networks — protein-protein interactions, gene regulatory networks,
metabolic pathways, co-expression networks — are naturally represented as graphs.
The graph Laplacian matrix bridges graph structure and linear algebra: its
eigenvectors reveal communities (functional modules), and its eigenvalues
quantify network connectivity.

---
## Step 2 — Intuition

**Adjacency matrix:** $A_{ij} = 1$ if there's an edge between nodes $i$ and $j$, else 0.

**Degree matrix:** $D_{ii} = \sum_j A_{ij}$ (number of connections of node $i$).

**Graph Laplacian:** $L = D - A$. Why subtract? Because $L\mathbf{1} = \mathbf{0}$
(constant vector is an eigenvector with eigenvalue 0). The Laplacian measures
how much a signal on the graph differs from its neighbors.

**Fiedler vector:** the eigenvector of $L$ corresponding to the second-smallest
eigenvalue (first nonzero). Positive entries → one community; negative entries → another.
It cuts the graph at its weakest connection.

---
## Step 3 — Biological Background

**Protein-protein interaction (PPI) networks:**
- Nodes = proteins; edges = physical interaction
- Highly connected nodes (hubs) are often essential genes — removing them disrupts
  the network (scale-free property)
- Community structure = protein complexes or functional modules

**Gene regulatory networks (GRN):**
- Directed edges: TF → target gene
- Network motifs (feedforward loops, feedback loops) are identified by their
  adjacency matrix substructure

**Spectral clustering of expression data:**
- Build a k-NN graph of samples (edge between two samples if they're among each
  other's k nearest neighbors in expression space)
- Laplacian eigenvectors cluster the samples — this is what Seurat's Louvain
  and Leiden clustering build upon

---
## Step 4 — Mathematical Explanation

**Adjacency matrix:** $A \in \{0,1\}^{n \times n}$ (undirected: symmetric).

**Degree matrix:** $D = \text{diag}(A\mathbf{1})$, where $\mathbf{1}$ is the all-ones vector.

**Graph Laplacian:** $L = D - A$.

**Properties of $L$:**
- Symmetric positive semi-definite (for undirected graphs)
- All eigenvalues $\lambda_i \geq 0$
- $\lambda_1 = 0$ always (eigenvector = $\mathbf{1}/\sqrt{n}$)
- Number of zero eigenvalues = number of connected components
- $\lambda_2 > 0$ iff the graph is connected; $\lambda_2$ is the **algebraic connectivity**

**Normalized Laplacian:** $\mathcal{L} = D^{-1/2} L D^{-1/2} = I - D^{-1/2}AD^{-1/2}$.
Used in spectral clustering to correct for degree heterogeneity (hub nodes).

**Spectral clustering algorithm (2 clusters):**
1. Compute graph Laplacian $L$
2. Find Fiedler vector $\mathbf{v}_2$ (eigenvector for $\lambda_2$)
3. Split: nodes with $v_{2,i} > 0$ go to cluster A; $v_{2,i} < 0$ to cluster B

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import networkx as nx
except ImportError:
    raise ImportError("Run: pip install networkx")

In [ ]:
# Cell 6.1 — Build adjacency and Laplacian from scratch
def build_laplacian(adj: np.ndarray) -> tuple:
    """
    Compute degree matrix D and graph Laplacian L = D - A.

    Parameters
    ----------
    adj : symmetric adjacency matrix

    Returns
    -------
    D : degree matrix
    L : graph Laplacian
    """
    degrees = adj.sum(axis=1)
    D = np.diag(degrees)
    L = D - adj
    return D, L

# Toy PPI network: two protein complexes loosely connected
rng = np.random.default_rng(42)
n1, n2 = 6, 6   # proteins in each complex
n = n1 + n2

# Dense intra-complex edges
A = np.zeros((n, n))
for i in range(n1):
    for j in range(i+1, n1):
        if rng.random() < 0.7:
            A[i, j] = A[j, i] = 1
for i in range(n1, n):
    for j in range(i+1, n):
        if rng.random() < 0.7:
            A[i, j] = A[j, i] = 1
# Sparse inter-complex edges
A[2, n1+1] = A[n1+1, 2] = 1   # one bridge

D, L = build_laplacian(A)

print(f"Adjacency matrix shape: {A.shape}")
print(f"Node degrees: {np.diag(D).astype(int)}")
print(f"L symmetric: {np.allclose(L, L.T)}")
print(f"L @ 1 = 0: {np.allclose(L @ np.ones(n), 0)}")

In [ ]:
# Cell 6.2 — Spectral analysis: Fiedler vector for community detection
eigenvalues_L, eigenvectors_L = np.linalg.eigh(L)

print(f"Eigenvalues of L (first 5): {eigenvalues_L[:5].round(4)}")
print(f"Number of zero eigenvalues (connected components): {(eigenvalues_L < 1e-8).sum()}")

fiedler_value  = eigenvalues_L[1]   # algebraic connectivity (λ₂)
fiedler_vector = eigenvectors_L[:, 1]

print(f"\nFiedler value λ₂ = {fiedler_value:.4f}  (larger = better connected)")
print(f"Fiedler vector: {fiedler_vector.round(3)}")

# Community assignment: sign of Fiedler vector
cluster_spectral = (fiedler_vector > 0).astype(int)
true_labels = np.array([0]*n1 + [1]*n2)

# Accuracy (accounting for label flip)
acc = max(
    np.mean(cluster_spectral == true_labels),
    np.mean(cluster_spectral == 1 - true_labels)
)
print(f"\nSpectral clustering accuracy: {acc:.2%}  (true communities: {n1}+{n2} nodes)")

In [ ]:
# Cell 6.3 — NetworkX: scale-free PPI network
G_sf = nx.barabasi_albert_graph(50, 2, seed=42)   # scale-free (Barabási-Albert)
A_sf = nx.to_numpy_array(G_sf)
D_sf, L_sf = build_laplacian(A_sf)
lam_sf, _ = np.linalg.eigh(L_sf)

degrees_sf = np.array([d for _, d in G_sf.degree()])
print(f"Scale-free network: n=50, edges={G_sf.number_of_edges()}")
print(f"Degree range: min={degrees_sf.min()}, max={degrees_sf.max()}, mean={degrees_sf.mean():.1f}")
print(f"Algebraic connectivity: {lam_sf[1]:.4f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — PPI network, Laplacian, Fiedler vector, degree distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Draw the toy PPI network colored by spectral clustering
ax = axes[0]
G_toy = nx.from_numpy_array(A)
pos = nx.spring_layout(G_toy, seed=42)
node_colors = ['steelblue' if c == 0 else 'tomato' for c in cluster_spectral]
nx.draw(G_toy, pos, ax=ax, node_color=node_colors, node_size=200, width=0.8,
        with_labels=True, font_size=8, font_color='white')
ax.set_title('PPI network: spectral clustering\n(blue/red = detected modules)')

# Panel 2: Fiedler vector values per node
ax = axes[1]
node_idx = np.arange(n)
ax.bar(node_idx, fiedler_vector,
       color=['steelblue' if v > 0 else 'tomato' for v in fiedler_vector])
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Node (protein)'); ax.set_ylabel('Fiedler vector value')
ax.set_title('Fiedler vector: sign → community\n(0 = inter-complex bridge node)')

# Panel 3: Degree distribution of scale-free network
ax = axes[2]
unique_deg, counts = np.unique(degrees_sf, return_counts=True)
ax.loglog(unique_deg, counts, 'o', color='steelblue', ms=6)
ax.set_xlabel('Degree k'); ax.set_ylabel('Count')
ax.set_title('Scale-free PPI: power-law degree distribution')

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `build_laplacian(adj)` from scratch. Verify that $L\mathbf{1} = \mathbf{0}$
   and that all eigenvalues of $L$ are ≥ 0.
2. What does the number of zero eigenvalues of $L$ equal? Add an isolated node to the
   toy network and verify the answer.
3. What is the normalized Laplacian $\mathcal{L} = D^{-1/2}LD^{-1/2}$? Implement it
   and compare its eigenvalue spectrum to $L$'s spectrum.
4. In a co-expression network built from expression data, an edge exists between two
   genes if their Pearson correlation > 0.8. The Laplacian Fiedler vector then
   splits genes into two communities. What biological structure might these communities
   correspond to?

---
## Quiz — Active Recall

1. Write $L = D - A$. What are $D$ and $A$?
2. Why does $L\mathbf{1} = \mathbf{0}$? Show the algebra.
3. What is the Fiedler vector? How is it used for community detection?
4. What does algebraic connectivity ($\lambda_2$) measure about a network?
5. A scale-free network has a power-law degree distribution. What does this imply
   about hub proteins in a PPI network?

---
## Reflection

**Date completed:** ____________________

> *[Could you explain the Laplacian and Fiedler vector to a systems biologist? What's the biological interpretation of spectral clustering?]*

---
*Next: `08_linear_regression_as_linear_algebra.ipynb`*